In [ ]:
from PIL import Image
import pytesseract
import re

imagen = "/musical/1968/Screenshot_2026-01-16-20-26-02-396_com.facebook.katana.jpg"
salida = "canciones_limpias.txt"

#texto_ocr = pytesseract.image_to_string(Image.open(imagen), lang="spa")
texto_ocr = pytesseract.image_to_string(Image.open(imagen))

patrones = [
    r'["“](.+?)["”]\s*[-—=]\s*(.+)',
    r'(.+?)\s*[-—=]\s*(.+)',
    r'(.+?)\s{2,}(.+)'
]

def limpiar_titulo(t):
    return t.strip().strip('"“”')

def limpiar_autor(a):
    return a.strip()

def es_valido(titulo, autor):
    blacklist = ["billboard", "gratis", "excel", "segui", "nicko", "based", "chart"]
    texto = f"{titulo} {autor}".lower()
    if any(pal in texto for pal in blacklist):
        return False
    if not re.search(r'[a-zA-Z]', titulo) or not re.search(r'[a-zA-Z]', autor):
        return False
    if any(sym in autor for sym in ["@", "|"]):
        return False
    return True

resultados = []
descartados = []

for linea in texto_ocr.splitlines():
    linea = linea.strip()
    linea = re.sub(r'^[\d\.\)\(]+', '', linea).strip()

    for patron in patrones:
        match = re.search(patron, linea)
        if match:
            titulo = limpiar_titulo(match.group(1))
            autor = limpiar_autor(match.group(2))
            if es_valido(titulo, autor):
                resultados.append(f"{titulo} - {autor}")
            else:
                descartados.append(f"{titulo} - {autor}")
            break

resultados = sorted(set(resultados))

with open(salida, "w", encoding="utf-8") as f:
    f.write("\n".join(resultados))

print("Archivo generado:", salida)
print("\n--- Líneas descartadas ---")
for d in descartados:
    print(d)


TesseractError: (1, 'Error opening data file C:\\Users\\marioL\\AppData\\Local\\Programs\\Tesseract-OCR/tessdata/spa.traineddata Please make sure the TESSDATA_PREFIX environment variable is set to your "tessdata" directory. Failed loading language \'spa\' Tesseract couldn\'t load any languages! Could not initialize tesseract.')